In [16]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, QuantoConfig
from torch.nn.utils import prune
import time

In [17]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
quantization_config = QuantoConfig(weights="int8")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    quantization_config=quantization_config
)

In [18]:
def llama_test(input_text: str):
    inputs = tokenizer(input_text, return_tensors="pt")
    start_time = time.time()

    prompt = f"User: {input_text}\nAssistant: Tolong jawab singkat kurang dari 20 kata."

    # Tokenize input and generate response
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        inputs.input_ids,
        max_length=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    end_time = time.time()
    inference_time = end_time - start_time
    print(f"Inference Time: {inference_time:.4f} seconds")
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    response = response.split("Assistant:")[-1].strip()
    print(response)

In [19]:
llama_test("Apa Ibukota Indonesia?")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Inference Time: 23.8153 seconds
Tolong jawab singkat kurang dari 20 kata. Indonesia adalah negara di Asia Tenggara. Ibukotanya adalah Jakarta.


In [7]:
def prune_model_layer(layer, amount=0.2):
    prune.l1_unstructured(layer, name="weight", amount=amount)

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear): 
        prune_model_layer(module, amount=0.2)

for module in model.modules():
    if isinstance(module, torch.nn.Linear):
        prune.remove(module, "weight")

torch.save(model.state_dict(), "llama_pruned.pth")

In [20]:
model.load_state_dict(torch.load("llama_pruned.pth"))

C:\Users\kimbe\AppData\Local\Temp\ipykernel_92592\1373804117.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("llama_pruned.pth"))


<All keys matched successfully>

In [30]:
llama_test("Ceritakan legenda 'Malin Kundang' secara singkat.")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Inference Time: 136.4635 seconds
Tolong jawab singkat kurang dari 20 kata. Malin Kundang adalah cerita rakyat Indonesia yang berlatar masa lalu, menceritakan kisah seorang lelorku yang bertanggung jawab dalam menyelesaikan sebuah masalah besar dengan cara yang tidak biasa, seperti mengambil tangan orang lain sebagai hukuman. Dia bertindak untuk menjadikan dirinya sebagai "Malin Kundang", sementara itu, lelorku lainnya menjadi "Malin Kundang". Kita harus mengingat bahwa orang-orang di desa di


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig
from transformers.utils import WEIGHTS_NAME, CONFIG_NAME
import torch
import os
import torch
from torch.nn.utils import prune
import time

# Initialize and quantize the model
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    quantization_config=QuantoConfig(weights="int8"),
)

# Prune the model
def prune_model_layer(layer, amount=0.2):
    prune.l1_unstructured(layer, name="weight", amount=amount)

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear): 
        prune_model_layer(module, amount=0.2)

for module in model.modules():
    if isinstance(module, torch.nn.Linear):
        prune.remove(module, "weight")

# Save the model in Hugging Face format
output_dir = "LLama_QTP"
os.makedirs(output_dir, exist_ok=True)

# Save tokenizer and model
tokenizer.save_pretrained(output_dir)
model.save_pretrained(output_dir)